In [1]:
#!pip install Sastrawi
#!pip install emoji
#!pip install nltk

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import emoji
import nltk
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
from sklearn.utils import resample


try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [3]:
df = pd.read_csv('data_review_roblox.csv')
df.head(10)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,80b7b59c-021c-4686-b489-c83f91aa4eff,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Semenjak viral server Mount dan banyak yg join...,1,119,2.694.983,2025-10-14 13:55:53,NaN,NaN,2.694.983
1,f8ea2835-6a40-4ac7-8ee8-5f54ea4df454,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"game nya bagus, dan seru.. cuma bisa minta tol...",4,4696,2.693.960,2025-10-10 20:55:15,NaN,NaN,2.693.960
2,ec58b3f1-70a2-4aa6-bba7-cff7ae2b9ec6,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,minnnn tolong kerjasamanya minnn 🤗🤗 sejauh ini...,3,32,2.693.960,2025-10-15 21:27:16,NaN,NaN,2.693.960
3,dedfd44d-faa6-4f34-8707-d68414200c28,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Udah dari lama denger tentang roblox, tapi juj...",5,744,2.694.983,2025-10-15 16:55:31,NaN,NaN,2.694.983
4,155a7030-22b4-4015-a6ce-b277c06a7e81,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"sebenernya Udah cukup memuaskan, tapi ada 1 ha...",3,22,2.694.983,2025-10-14 10:51:22,NaN,NaN,2.694.983
5,59dc9f43-7a25-4b8a-ae3b-fe7ec0b49845,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aku suka bangat game ini 😚🤍 di dalam banyak ma...,5,616,2.694.983,2025-10-15 18:50:07,NaN,NaN,2.694.983
6,47e5ebbb-3bda-404e-be36-6bfa81b5984f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"BAGUS BANGET WOY. minusnya tombol lompat nya, ...",5,4247,2.694.983,2025-10-11 21:55:36,NaN,NaN,2.694.983
7,b12610fa-45c5-4658-8060-2849437ca766,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Banyak banget game yang bisa di mainkan, seru ...",5,2837,2.693.960,2025-10-10 08:57:20,NaN,NaN,2.693.960
8,1fc05f28-a1cd-4518-8739-0557697074e3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"game nya seru, bagus. tapi kenapa ya akhir akh...",5,1355,2.693.960,2025-10-10 17:43:11,NaN,NaN,2.693.960
9,0075c608-7b3f-4597-b472-7fb01f030a5a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Sering banget DC ( disconnected ) padahal jari...,5,21,2.694.983,2025-10-16 00:00:44,NaN,NaN,2.694.983


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72000 entries, 0 to 71999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   reviewId              72000 non-null  object 
 1   userName              72000 non-null  object 
 2   userImage             72000 non-null  object 
 3   content               71999 non-null  object 
 4   score                 72000 non-null  int64  
 5   thumbsUpCount         72000 non-null  int64  
 6   reviewCreatedVersion  65813 non-null  object 
 7   at                    72000 non-null  object 
 8   replyContent          0 non-null      float64
 9   repliedAt             0 non-null      float64
 10  appVersion            65813 non-null  object 
dtypes: float64(2), int64(2), object(7)
memory usage: 6.0+ MB


In [5]:
df = df.drop_duplicates(subset=['reviewId'])
columns_to_drop = ['replyContent', 'repliedAt', 'userImage', 'reviewCreatedVersion', 'userName', 'thumbsUpCount', 'appVersion', 'at']
df = df.drop(columns= columns_to_drop)

df = df.dropna(subset=['content'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71999 entries, 0 to 71999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   reviewId  71999 non-null  object
 1   content   71999 non-null  object
 2   score     71999 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.2+ MB


In [6]:
df.isnull().sum() 

reviewId    0
content     0
score       0
dtype: int64

In [7]:
# Lihat sample content asli
print("Total reviews:", len(df))
print("Sample 5 reviews pertama:")
for i in range(min(5, len(df))):
    print(f"Score: {df['score'].iloc[i]}")
    print(f"Content: {df['content'].iloc[i]}")

Total reviews: 71999
Sample 5 reviews pertama:
Score: 1
Content: Semenjak viral server Mount dan banyak yg join, apk Roblox jadi sering update dan ukuran apknya jadi tambah besar dan berat :( sering disconnect juga, item di marketplace juga jadi mahal dan banyak item yang duplikat. Gamenya juga lebih banyak memakan kuota internet jadi lebih boros dari tahun-tahun sebelumnya, menurutku sering update bukan bertambah baik,tapi malah ada penurunan :(, aku sudah bermain roblox sangat lama jadi merasakan perbedaannya,"ada server yg mengandung aktifitas dewasa" 😫☠️
Score: 4
Content: game nya bagus, dan seru.. cuma bisa minta tolong ga? bug nya di ilangin.. aku kira aku doang yg ngebug ternyata semua, aku uda hapus²in semua data²nya ttp ngebug gmn yah? tolong ya ilangin bug nya.. tiap masuk map/server pasti selalu jaringan, padahal uda pake wifi kadang juga data tapi tetep ngelag..
Score: 3
Content: minnnn tolong kerjasamanya minnn 🤗🤗 sejauh ini aplikasinya sangat sangat bagussss dan menyenang

In [8]:
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopwords = [
    'min', 'minnn', 'admin', 'tolong', 'saya', 'nya', 'ya', 'yg', 'dgn', 'd', 
    'sdh', 'udah', 'ga', 'gak', 'nggak', 'tdk', 'tidak', 'bang', 'banget',
    'sih', 'deh', 'dong', 'yaa', 'yah', 'sama', 'dengan', 'dari', 'di', 'ke',
    'yang', 'untuk', 'pada', 'dalam', 'game', 'aplikasi', 'app', 'review', 'roblox'
]

def clean_stopwords(text):
    if pd.isna(text) or text == '':
        return text
    text = emoji.replace_emoji(text, replace=' ') #emoji removal
    text = str(text).lower() 
    text = re.sub(r'@\w+\s*', '', text) #remove mentions
    text = re.sub(r'#\w+\s*', '', text) #remove hashtags
    text = re.sub(r'(.)\1{2,}', ' ', text) #remove repetitive characters
    text = re.sub(r'[^\w\s]', ' ', text) #remove punctuation
    text = re.sub(r'\d+', ' ', text) #remove numbers
    text = re.sub(r'\s+', ' ', text).strip() #remove extra spaces

    return text

def preprocess_roblox_content(text):
   
    # Step 1: Basic cleaning
    cleaned = clean_stopwords(text)
    if not cleaned:
        return ""
    
    # Step 2: tokenization 
    tokens = cleaned.split() #
    # Step 3: removal stopwords
    stopped_tokens = []
    for token in tokens: 
        cleaned_token = stopword_remover.remove(token)
        if cleaned_token.strip():
            stopped_tokens.append(cleaned_token)
    #step 4: custom stopword removal
    custom_stopped_tokens = [token for  token in stopped_tokens if token not in stopwords and len(token) > 2]

    # Step 5: stemming
    stemmed_tokens = [stemmer.stem(token) for token in custom_stopped_tokens]

    #step 6: final cleaning dan join kembali
    final_teks = ' '.join(stemmed_tokens)
    final_teks = re.sub(r'\s+', ' ', final_teks).strip()

    if len(final_teks.split())<2:
        return ""

    return final_teks


In [9]:
# Test fungsi baru
test_text = "Game Roblox ini sangat seru banget! Tapi kadang lag nih."
result = preprocess_roblox_content(test_text)
print("Hasil:", result)

Hasil: sangat seru kadang lag nih


In [10]:
# Terapkan preprocessing ke seluruh dataset
df['cleaned_content'] = df['content'].apply(preprocess_roblox_content)

for i in range(3):
    print(f"Original: {df['content'].iloc[i]}")
    print(f"Cleaned: {df['cleaned_content'].iloc[i]}")
    print("")

before_count = len(df)
df = df[df['cleaned_content'] != '']
after_count = len(df)
print(f"Data setelah cleaning: {after_count}/{before_count}")

Original: Semenjak viral server Mount dan banyak yg join, apk Roblox jadi sering update dan ukuran apknya jadi tambah besar dan berat :( sering disconnect juga, item di marketplace juga jadi mahal dan banyak item yang duplikat. Gamenya juga lebih banyak memakan kuota internet jadi lebih boros dari tahun-tahun sebelumnya, menurutku sering update bukan bertambah baik,tapi malah ada penurunan :(, aku sudah bermain roblox sangat lama jadi merasakan perbedaannya,"ada server yg mengandung aktifitas dewasa" 😫☠️
Cleaned: semenjak viral server mount banyak join apk jadi sering update ukur apknya jadi tambah besar berat sering disconnect item marketplace jadi mahal banyak item duplikat gamenya lebih banyak makan kuota internet jadi lebih boros tahun tahun belum turut sering update bukan tambah baik malah turun aku main sangat lama jadi rasa beda server kandung aktifitas dewasa

Original: game nya bagus, dan seru.. cuma bisa minta tolong ga? bug nya di ilangin.. aku kira aku doang yg ngebug terny

In [11]:
# LABELING BERDASARKAN KOLOM SCORE
def label_sentiment(score):
    if score >= 4:
        return 'positif'
    elif score == 3:
        return 'netral'
    else:  
        return 'negatif'

# Terapkan labeling ke dataset
df['sentiment'] = df['score'].apply(label_sentiment)

# Tampilkan hasil
print("Distribusi Sentimen:")
print(df['sentiment'].value_counts())
print("\nPersentase:")
print(df['sentiment'].value_counts(normalize=True).round(3) * 100)

Distribusi Sentimen:
sentiment
positif    55119
negatif    10970
netral      5450
Name: count, dtype: int64

Persentase:
sentiment
positif    77.0
negatif    15.3
netral      7.6
Name: proportion, dtype: float64


In [12]:
# Pisahkan data berdasarkan kelas
df_positif = df[df['sentiment'] == 'positif']
df_negatif = df[df['sentiment'] == 'negatif']
df_netral = df[df['sentiment'] == 'netral']

print("Sebelum balancing:")
print(f"Positif: {len(df_positif)}")
print(f"Negatif: {len(df_negatif)}")
print(f"Netral: {len(df_netral)}")

# OPSI 1: OVERSAMPLING KELAS MINORITAS
df_negatif_oversampled = resample(df_negatif,
                                 replace=True,
                                 n_samples=25000,
                                 random_state=42)

df_netral_oversampled = resample(df_netral,
                                replace=True,
                                n_samples=25000,
                                random_state=42)

# Gabungkan kembali
df_balanced = pd.concat([df_positif, df_negatif_oversampled, df_netral_oversampled])

print("\nSetelah balancing:")
print(df_balanced['sentiment'].value_counts())

Sebelum balancing:
Positif: 55119
Negatif: 10970
Netral: 5450

Setelah balancing:
sentiment
positif    55119
negatif    25000
netral     25000
Name: count, dtype: int64


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85
)

X = vectorizer.fit_transform(df_balanced['cleaned_content'])
y = df_balanced['sentiment']

print(f"Shape features: {X.shape}")

# Split data dengan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

# Cek distribusi setelah split
print("\nDistribusi Training:")
print(y_train.value_counts())
print("\nDistribusi Testing:")
print(y_test.value_counts())

Shape features: (105119, 8000)
Training set: 84095 samples
Testing set: 21024 samples

Distribusi Training:
sentiment
positif    44095
negatif    20000
netral     20000
Name: count, dtype: int64

Distribusi Testing:
sentiment
positif    11024
negatif     5000
netral      5000
Name: count, dtype: int64


In [14]:
#!pip install tensorflow

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Convert to categorical
y_train_categorical = to_categorical(y_train_encoded)
y_test_categorical = to_categorical(y_test_encoded)

print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# Build model
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model summary:")
model.summary()

Label mapping: {'negatif': 0, 'netral': 1, 'positif': 2}


c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     4,096,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,261,123 (16.25 MB)

 Trainable params: 4,261,123 (16.25 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

# Training
history = model.fit(
    X_train, y_train_categorical,
    epochs=30,
    batch_size=64,
    validation_data=(X_test, y_test_categorical),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

Epoch 1/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 75s 53ms/step - accuracy: 0.7527 - loss: 0.5995 - val_accuracy: 0.8530 - val_loss: 0.4004
Epoch 2/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 64s 49ms/step - accuracy: 0.9041 - loss: 0.2760 - val_accuracy: 0.9105 - val_loss: 0.2603
Epoch 3/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 69s 52ms/step - accuracy: 0.9518 - loss: 0.1429 - val_accuracy: 0.9261 - val_loss: 0.2343
Epoch 4/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.9698 - loss: 0.0900 - val_accuracy: 0.9336 - val_loss: 0.2313
Epoch 5/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 67s 51ms/step - accuracy: 0.9782 - loss: 0.0653 - val_accuracy: 0.9363 - val_loss: 0.2379
Epoch 6/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 64s 48ms/step - accuracy: 0.9841 - loss: 0.0482 - val_accuracy: 0.9404 - val_loss: 0.2510
Epoch 7/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.9870 - loss: 0.0371 - val_accuracy: 0.9401 - val_loss: 0.2637


In [22]:
# Evaluate model
train_loss, train_accuracy = model.evaluate(X_train, y_train_categorical, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test, y_test_categorical, verbose=0)

print(f"Akurasi Training: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Akurasi Testing: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

Akurasi Training: 0.9920 (99.20%)
Akurasi Testing: 0.9336 (93.36%)


In [18]:
# SKEMA 2: RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 50)
print("RANDOM FOREST + TF-IDF")
print("=" * 50)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Akurasi Random Forest: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")

RANDOM FOREST + TF-IDF
Akurasi Random Forest: 0.9390 (93.90%)


In [19]:
from sklearn.svm import SVC

print("\n" + "=" * 50)
print("SVM + TF-IDF")
print("=" * 50)

svm_model = SVC(
    kernel='linear',
    C=1.0,
    random_state=42
)

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)

print(f"Akurasi SVM: {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")


SVM + TF-IDF
Akurasi SVM: 0.7839 (78.39%)


In [25]:
from sklearn.metrics import accuracy_score

# Evaluasi model lainnya
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

svm_pred = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)

print("=" * 70)
print("SUMMARY HASIL 3 SKEMA PERCOBAAN")
print("=" * 70)

results = {
    "Deep Learning Neural Network": test_accuracy,  # sudah didefinisikan
    "Random Forest": rf_accuracy, 
    "SVM": svm_accuracy
}

for model_name, accuracy in results.items():
    status = "TERPENUHI" if accuracy >= 0.85 else "TIDAK TERPENUHI"
    print(f"{model_name:30} : {accuracy:.4f} ({accuracy*100:.2f}%) - {status}")

SUMMARY HASIL 3 SKEMA PERCOBAAN
Deep Learning Neural Network   : 0.9336 (93.36%) - TERPENUHI
Random Forest                  : 0.9390 (93.90%) - TERPENUHI
SVM                            : 0.7839 (78.39%) - TIDAK TERPENUHI


In [26]:
import joblib

# Simpan semua model
model.save('model_deep_learning.h5')
joblib.dump(rf_model, 'model_random_forest.pkl')
joblib.dump(svm_model, 'model_svm.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

print("Semua model berhasil disimpan:")
print("- model_deep_learning.h5")
print("- model_random_forest.pkl")
print("- model_svm.pkl") 
print("- tfidf_vectorizer.pkl")
print("- label_encoder.pkl")

Semua model berhasil disimpan:
- model_deep_learning.h5
- model_random_forest.pkl
- model_svm.pkl
- tfidf_vectorizer.pkl
- label_encoder.pkl


In [31]:
def sentiment_analysis_inference(text, model_type='deep_learning'):
    """
    Fungsi inference untuk analisis sentimen
    """
    # Preprocessing
    cleaned_text = preprocess_roblox_content(text)
    if not cleaned_text:
        return "netral", 0.0
    
    # Transform features
    features = vectorizer.transform([cleaned_text])
    
    # Predict berdasarkan model yang dipilih
    if model_type == 'deep_learning':
        prediction = model.predict(features)
        predicted_class_idx = prediction.argmax()
        confidence = prediction.max()
    elif model_type == 'random_forest':
        prediction = rf_model.predict_proba(features)
        predicted_class_idx = prediction.argmax()
        confidence = prediction.max()
    elif model_type == 'svm':
        prediction = svm_model.decision_function(features)
        predicted_class_idx = prediction.argmax()
        confidence = prediction.max()
    
    predicted_class = label_encoder.inverse_transform([predicted_class_idx])[0]
    
    return predicted_class, confidence

print("DEMO INFERENCE")
print("=" * 60)

# Test cases untuk menunjukkan hasil inference
test_samples = [
    "game ini sangat bagus dan seru banget recommended deh",
    "aplikasi sering error dan lag terus frustrating banget",
    "cukup lumayan tapi biasa aja tidak spesial",
    "grafis keren gameplay smooth worth it untuk dimainkan",
    "server sering down dan banyak bug perlu perbaikan"
]

print("Hasil prediksi sentimen:\n")
for i, text in enumerate(test_samples, 1):
    # Gunakan model deep learning (yang terbaik)
    sentiment, confidence = sentiment_analysis_inference(text, 'deep_learning')
    
    print(f"{i}. TEXT: '{text}'")
    print(f"   SENTIMEN: {sentiment.upper()} (confidence: {confidence:.2f})")
    print()

DEMO INFERENCE
Hasil prediksi sentimen:

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1. TEXT: 'game ini sangat bagus dan seru banget recommended deh'
   SENTIMEN: POSITIF (confidence: 0.91)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
2. TEXT: 'aplikasi sering error dan lag terus frustrating banget'
   SENTIMEN: NEGATIF (confidence: 0.97)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step
3. TEXT: 'cukup lumayan tapi biasa aja tidak spesial'
   SENTIMEN: NETRAL (confidence: 0.58)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step
4. TEXT: 'grafis keren gameplay smooth worth it untuk dimainkan'
   SENTIMEN: POSITIF (confidence: 1.00)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step
5. TEXT: 'server sering down dan banyak bug perlu perbaikan'
   SENTIMEN: NEGATIF (confidence: 0.98)



In [48]:
# Cell khusus untuk screenshot hasil inference
demo_texts = [
    "roblox game yang amazing sangat menghibur",
    "saya kecewa dengan update terakhir jadi lag",
    "aplikasi standar tidak istimewa tapi bisa dipakai"
]

for text in demo_texts:
    sentiment, conf = sentiment_analysis_inference(text)
    print(f"Input: {text}")
    print(f"Output: {sentiment}")
    print(f"Confidence: {conf:.2f}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
Input: roblox game yang amazing sangat menghibur
Output: positif
Confidence: 1.00
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
Input: saya kecewa dengan update terakhir jadi lag
Output: positif
Confidence: 0.50
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
Input: aplikasi standar tidak istimewa tapi bisa dipakai
Output: negatif
Confidence: 0.51
--------------------------------------------------


In [49]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
